# Simple OTel Emitter (Local -> Unity Catalog)

This notebook is a simplified version with no external telemetry config file.

## What this notebook does
- Loads Databricks + table routing config from `.env`
- Configures OTLP exporters for traces, logs, and metrics
- Provides separate cells to emit each signal type
- Flushes and optionally verifies row counts via SQL Warehouse

## Required `.env` keys
- `DATABRICKS_HOST`
- `DATABRICKS_TOKEN`
- `DATABRICKS_CATALOG`
- `DATABRICKS_SCHEMA`
- `OTEL_SERVICE_NAME`
- `OTEL_SPANS_TABLE`
- `OTEL_LOGS_TABLE`
- `OTEL_METRICS_TABLE`

Optional:
- `DATABRICKS_WAREHOUSE_ID`


In [ ]:
# Install dependencies into the active notebook kernel
import subprocess
import sys

packages = [
    "opentelemetry-api",
    "opentelemetry-sdk",
    "opentelemetry-exporter-otlp-proto-http",
    "python-dotenv",
    "databricks-sql-connector",
]

base_cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
try:
    subprocess.check_call(base_cmd)
except subprocess.CalledProcessError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--user",
            "--break-system-packages",
            *packages,
        ]
    )

print(f"Using Python: {sys.executable}")
print("Dependencies installed")

In [ ]:
# Load .env and validate required config
from pathlib import Path
import os
import site
import sys

from dotenv import load_dotenv

if site.ENABLE_USER_SITE and site.getusersitepackages() not in sys.path:
    sys.path.append(site.getusersitepackages())

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
load_dotenv(project_root / ".env")

required_env = [
    "DATABRICKS_HOST",
    "DATABRICKS_TOKEN",
    "DATABRICKS_CATALOG",
    "DATABRICKS_SCHEMA",
    "OTEL_SERVICE_NAME",
    "OTEL_SPANS_TABLE",
    "OTEL_LOGS_TABLE",
    "OTEL_METRICS_TABLE",
]
missing = [k for k in required_env if not os.getenv(k)]
if missing:
    raise ValueError(f"Missing required .env keys: {', '.join(missing)}")

DATABRICKS_HOST = os.environ["DATABRICKS_HOST"].rstrip("/")
DATABRICKS_TOKEN = os.environ["DATABRICKS_TOKEN"]
DATABRICKS_CATALOG = os.environ["DATABRICKS_CATALOG"]
DATABRICKS_SCHEMA = os.environ["DATABRICKS_SCHEMA"]
OTEL_SERVICE_NAME = os.environ["OTEL_SERVICE_NAME"]
OTEL_SPANS_TABLE = os.environ["OTEL_SPANS_TABLE"]
OTEL_LOGS_TABLE = os.environ["OTEL_LOGS_TABLE"]
OTEL_METRICS_TABLE = os.environ["OTEL_METRICS_TABLE"]

print("Loaded config:")
print(f"  workspace: {DATABRICKS_HOST}")
print(f"  schema:    {DATABRICKS_CATALOG}.{DATABRICKS_SCHEMA}")
print(f"  service:   {OTEL_SERVICE_NAME}")
print(f"  spans:     {OTEL_SPANS_TABLE}")
print(f"  logs:      {OTEL_LOGS_TABLE}")
print(f"  metrics:   {OTEL_METRICS_TABLE}")

In [ ]:
# Configure exporters and providers
import logging
import time
import urllib.error
import urllib.request

from opentelemetry import _logs as otel_logs
from opentelemetry import metrics, trace
from opentelemetry.exporter.otlp.proto.http._log_exporter import OTLPLogExporter
from opentelemetry.exporter.otlp.proto.http.metric_exporter import OTLPMetricExporter
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk._logs import LoggerProvider, LoggingHandler
from opentelemetry.sdk._logs.export import SimpleLogRecordProcessor
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

TRACES_ENDPOINT = f"{DATABRICKS_HOST}/api/2.0/tracing/otel/v1/traces"
LOGS_ENDPOINT = f"{DATABRICKS_HOST}/api/2.0/tracing/otel/v1/logs"
METRICS_ENDPOINT = f"{DATABRICKS_HOST}/api/2.0/otel/v1/metrics"

RESOURCE = Resource.create(
    {
        "service.name": OTEL_SERVICE_NAME,
        "service.version": "0.1.0",
        "deployment.environment": "local",
    }
)

def make_headers(table_name: str):
    return {
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "X-Databricks-UC-Table-Name": table_name,
        "X-Databricks-Workspace-Url": DATABRICKS_HOST,
    }

def smoke_check(name: str, url: str, table_name: str):
    req = urllib.request.Request(
        url=url,
        method="POST",
        data=b"",
        headers={**make_headers(table_name), "Content-Type": "application/x-protobuf"},
    )
    try:
        urllib.request.urlopen(req, timeout=8)
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"{name} endpoint failed: {url} -> {e.code} {e.reason}") from e

smoke_check("traces", TRACES_ENDPOINT, OTEL_SPANS_TABLE)
smoke_check("logs", LOGS_ENDPOINT, OTEL_LOGS_TABLE)
smoke_check("metrics", METRICS_ENDPOINT, OTEL_METRICS_TABLE)

# Trace provider
span_exporter = OTLPSpanExporter(endpoint=TRACES_ENDPOINT, headers=make_headers(OTEL_SPANS_TABLE))
tracer_provider = TracerProvider(resource=RESOURCE)
tracer_provider.add_span_processor(SimpleSpanProcessor(span_exporter))
trace.set_tracer_provider(tracer_provider)
tracer = trace.get_tracer(OTEL_SERVICE_NAME)

# Log provider
log_exporter = OTLPLogExporter(endpoint=LOGS_ENDPOINT, headers=make_headers(OTEL_LOGS_TABLE))
logger_provider = LoggerProvider(resource=RESOURCE)
logger_provider.add_log_record_processor(SimpleLogRecordProcessor(log_exporter))
otel_logs.set_logger_provider(logger_provider)
otel_handler = LoggingHandler(level=logging.INFO, logger_provider=logger_provider)
otel_logger = logging.getLogger(OTEL_SERVICE_NAME)
otel_logger.handlers = []
otel_logger.addHandler(otel_handler)
otel_logger.setLevel(logging.INFO)

# Metric provider
metric_export_interval_ms = 5000
metric_exporter = OTLPMetricExporter(endpoint=METRICS_ENDPOINT, headers=make_headers(OTEL_METRICS_TABLE))
metric_reader = PeriodicExportingMetricReader(metric_exporter, export_interval_millis=metric_export_interval_ms)
meter_provider = MeterProvider(resource=RESOURCE, metric_readers=[metric_reader])
metrics.set_meter_provider(meter_provider)
meter = metrics.get_meter(OTEL_SERVICE_NAME)

request_counter = meter.create_counter("http.server.request_count", description="Total HTTP requests", unit="requests")
request_duration = meter.create_histogram("http.server.duration", description="HTTP request duration", unit="ms")

print("Exporters configured")
print(f"  traces endpoint:  {TRACES_ENDPOINT}")
print(f"    - pushing telemetry to {OTEL_SPANS_TABLE}")
print(f"  logs endpoint:    {LOGS_ENDPOINT}")
print(f"    - pushing telemetry to {OTEL_LOGS_TABLE}")
print(f"  metrics endpoint: {METRICS_ENDPOINT}")
print(f"    - pushing telemetry to {OTEL_METRICS_TABLE}")

## What "Traces" Means (Customer Journey Story)

A **trace** is the story of one request moving through your system.

In this demo, we emit:
- One parent step: `GET /api/v1/queries`
- One child step: `db.execute`
- Small "events" inside those steps (like milestones)

Plain English: this shows **how long work took** and **where time was spent** for one user request.

In [ ]:
# Emit traces (with events)
print("Emitting traces...")
with tracer.start_as_current_span(
    "GET /api/v1/queries",
    attributes={"http.method": "GET", "http.route": "/api/v1/queries", "http.status_code": 200},
) as parent:
    parent.add_event("query.parse", attributes={"query.length": 142})
    time.sleep(0.05)
    with tracer.start_as_current_span(
        "db.execute",
        attributes={"db.system": "databricks", "db.operation": "SELECT"},
    ) as child:
        child.add_event("rows.fetched", attributes={"row.count": 10})
        time.sleep(0.02)

print("Trace emission complete")

## What "Logs" Means (Business-Friendly Notes)

A **log** is a timestamped message about what happened.

In this demo, we emit:
- One success message
- One warning message (slow query)

Plain English: logs explain **what happened** and provide context for operations and support teams.

In [ ]:
# Emit logs
print("Emitting logs...")
otel_logger.info("Query completed successfully", extra={"query_id": "q-simple-1", "rows_returned": 42})
otel_logger.warning("Slow query observed", extra={"query_id": "q-simple-2", "duration_ms": 230.7})
print("Log emission complete")

## What "Metrics" Means (KPI / Dashboard Numbers)

A **metric** is a numeric signal you can chart over time.

In this demo, we emit:
- Request count (how many requests)
- Request duration (how fast/slow requests are)

Plain English: metrics are your **dashboard health numbers** for trend monitoring and alerting.

In [ ]:
# Emit metrics
print("Emitting metrics...")
request_counter.add(1, {"http.method": "GET", "http.route": "/api/v1/queries"})
request_duration.record(45.3, {"http.method": "GET", "http.route": "/api/v1/queries"})
request_counter.add(1, {"http.method": "POST", "http.route": "/api/v1/analysis"})
request_duration.record(230.7, {"http.method": "POST", "http.route": "/api/v1/analysis"})

print("Waiting for metric export interval...")
time.sleep(6)
print("Metric emission complete")

In [ ]:
# Flush and shutdown
for name, provider in [
    ("Traces", tracer_provider),
    ("Logs", logger_provider),
    ("Metrics", meter_provider),
]:
    provider.force_flush(timeout_millis=10000)
    print(f"{name} flushed")

tracer_provider.shutdown()
logger_provider.shutdown()
meter_provider.shutdown()
print("All providers shut down")

In [ ]:
# Optional verification via SQL Warehouse
warehouse_id = os.getenv("DATABRICKS_WAREHOUSE_ID", "").strip()
if not warehouse_id:
    print("DATABRICKS_WAREHOUSE_ID is empty; skipping verification query.")
else:
    from databricks import sql

    print("Running verification counts...")
    with sql.connect(
        server_hostname=DATABRICKS_HOST.replace("https://", ""),
        http_path=f"/sql/1.0/warehouses/{warehouse_id}",
        access_token=DATABRICKS_TOKEN,
    ) as conn:
        with conn.cursor() as cursor:
            for label, table_name in [
                ("spans", OTEL_SPANS_TABLE),
                ("logs", OTEL_LOGS_TABLE),
                ("metrics", OTEL_METRICS_TABLE),
            ]:
                cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
                count = cursor.fetchone()[0]
                print(f"  {label}: {table_name} -> {count}")